# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions


Paste the answer as Python in the answer code section below each question.


### Question 1

Join the `metadata_pl` and `ratings_pl` DataFrames in Polars, then calculate the average rating by `original_language` (separately).


In [1]:
import requests
import os

# Download the files
gcs_bucket_url = "https://storage.googleapis.com/su-artifacts/"
file_names = ["movies_metadata.csv", "ratings.csv"]

for name in file_names:
    if os.path.exists('../data/'+name):
        print (f"File: {name} already exist in your data folder. Skipping download.")
    else:
        full_url = gcs_bucket_url + name
        r = requests.get(full_url)
        with open("../data/" + name, "wb") as f:
            f.write(r.content)

File: movies_metadata.csv already exist in your data folder. Skipping download.
File: ratings.csv already exist in your data folder. Skipping download.


**Answer:**

In [20]:
import polars as pl

# 1. Load Data (as before)
metadata_pl = pl.scan_csv("../data/movies_metadata.csv", infer_schema_length=100000)
ratings_pl = pl.scan_csv("../data/ratings.csv", infer_schema_length=100000)


# Prepare ratings
ratings_prepared = ratings_pl.select(
    pl.col("movieId").cast(pl.Utf8),
    pl.col("rating")
)

# Prepare metadata
metadata_prepared = metadata_pl.with_columns(
    pl.col("id").cast(pl.Utf8),
    pl.col("original_language")
)

ratings_agg = (
    ratings_prepared
    .group_by("movieId")
    .agg(pl.mean("rating").alias("avg_rating"))
)
joined_df = metadata_prepared.join(
    ratings_agg,
    left_on="id",
    right_on="movieId",
    how="inner"
)
language_ratings = (
    joined_df
    .group_by("original_language")
    .agg(pl.mean("avg_rating").alias("avg_rating"))
    .sort("avg_rating", descending=True)
    .collect()
)

print(language_ratings)

shape: (57, 2)
┌───────────────────┬────────────┐
│ original_language ┆ avg_rating │
│ ---               ┆ ---        │
│ str               ┆ f64        │
╞═══════════════════╪════════════╡
│ ka                ┆ 3.754167   │
│ mr                ┆ 3.652174   │
│ kk                ┆ 3.610053   │
│ uk                ┆ 3.504316   │
│ eu                ┆ 3.5        │
│ …                 ┆ …          │
│ ar                ┆ 2.522222   │
│ null              ┆ 2.458333   │
│ tl                ┆ 2.433592   │
│ lo                ┆ 2.0        │
│ sl                ┆ 2.0        │
└───────────────────┴────────────┘


In [21]:
type(joined_df)

polars.lazyframe.frame.LazyFrame

### Question 2

Calculate the average total amount for vendors with at least 5 trips from the NYC Taxi Trip Data.

In [2]:
gcs_bucket_url = "https://storage.googleapis.com/su-artifacts/"
full_url = gcs_bucket_url + 'taxi_trip_data.csv'

# Check if file already exists and in that case skip download, otherwise download
if os.path.exists('../data/taxi_trip_data.csv'):
    print("File already exist.")
else:
    print("Downloading taxi_trip_data.csv...")
    print("Please note that this is a very large file (~1.5GB). Please make sure you have enough disk space and a stable internet connection.")
    print("This may take several minutes...")
    r = requests.get(full_url)
    with open("../data/" + 'taxi_trip_data.csv', "wb") as f:
        f.write(r.content)

Please note that this is a very large file (~1.5GB). Please make sure you have enough disk space and a stable internet connection.
This may take several minutes...


**Answer**

In [3]:
import polars as pl

# Read NYC Taxi data and calculate average total amount for vendors with >= 5 trips
result = (
    pl.scan_csv("../data/taxi_trip_data.csv")
    .group_by('vendor_id')
    .agg([
        pl.count('total_amount').alias('trip_count'),
        pl.mean('total_amount').alias('avg_total_amount')
    ])
    .filter(pl.col('trip_count') >= 5)
    .sort('avg_total_amount', descending=True)
    .collect()
)

In [4]:
print(result)

shape: (3, 3)
┌───────────┬────────────┬──────────────────┐
│ vendor_id ┆ trip_count ┆ avg_total_amount │
│ ---       ┆ ---        ┆ ---              │
│ i64       ┆ u32        ┆ f64              │
╞═══════════╪════════════╪══════════════════╡
│ 2         ┆ 6003561    ┆ 41.010683        │
│ 1         ┆ 3949867    ┆ 39.786292        │
│ 4         ┆ 46572      ┆ 38.650614        │
└───────────┴────────────┴──────────────────┘
